In [0]:
MY_ID = "Musarraf_7045"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

### Silver decisions
1. Trim txn_id on both sides before anything else (key normalization).
2. try_cast amount -> DECIMAL(14,2); ledger strips commas before casting.
   Gateway NULLs are kept as a break type, not dropped.
3. Dedupe each side separately on trimmed txn_id; counts printed and reported.
4. Match tolerance defined (applied in Gold): trimmed txn_id equal AND
   abs(gateway_amount - ledger_amount) <= 0.05. settlement_date = date(txn_ts)
   added to both Silver tables.
5. status mapped to a single `settled` boolean here, not in the Gold join.

##Silver gateway

In [0]:
from pyspark.sql import functions as F

bronze_g = spark.table(f"workspace.capstone_{MY_ID}.bronze_gateway")

silver_g = (bronze_g
    .withColumn("txn_id", F.trim(F.col("txn_id")))                      # Decision 1
    .withColumn("amount", F.expr("try_cast(amount as decimal(14,2))"))  # Decision 2
    .withColumn("settled", F.col("status") == F.lit("SUCCESS"))         # Decision 5
    .withColumn("settlement_date", F.to_date("txn_ts"))                 # Decision 4
)

gateway_before = silver_g.count()
silver_g = silver_g.dropDuplicates(["txn_id"])                          # Decision 3
gateway_after = silver_g.count()
gateway_nulls = silver_g.filter(F.col("amount").isNull()).count()

print("gateway:", gateway_before, "->", gateway_after, "| NULL amounts:", gateway_nulls)

silver_g.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_gateway")

gateway: 41468 -> 41208 | NULL amounts: 350


##Silver ledger

In [0]:
bronze_l = spark.table(f"workspace.capstone_{MY_ID}.bronze_ledger")

silver_l = (bronze_l
    .withColumn("txn_id", F.trim(F.col("txn_id")))                                       # Decision 1
    .withColumn("amount", F.expr("try_cast(replace(amount,',','') as decimal(14,2))"))    # Decision 2 — strip comma first
    .withColumn("settled", F.col("status") == F.lit("POSTED"))                            # Decision 5
    .withColumn("settlement_date", F.to_date("txn_ts"))                                   # Decision 4
)

ledger_before = silver_l.count()
silver_l = silver_l.dropDuplicates(["txn_id"])                                            # Decision 3
ledger_after = silver_l.count()
ledger_nulls = silver_l.filter(F.col("amount").isNull()).count()

print("ledger:", ledger_before, "->", ledger_after, "| NULL amounts:", ledger_nulls)

silver_l.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_ledger")

ledger: 41323 -> 41193 | NULL amounts: 0
